# A* Search Algorithm

This notebook implements the **A\* (A-star) search algorithm** on a weighted directed graph.

A\* combines the actual path cost `g(n)` with a heuristic estimate `h(n)` to find the optimal path from a start node to a goal node.

## 1. Imports

In [7]:
# Import Optional type hint for functions that may return None
from typing import Optional

## 2. Graph Class Definition

In [10]:
class Graph:
    """Represents a weighted directed graph with heuristic values for A* search."""

    def __init__(self, edges: list[tuple], heuristic: dict[str, float]) -> None:
        # Store the list of edges; each edge is a tuple (source, target, weight)
        self.edges: list[tuple] = edges
        # Store the heuristic values; maps each node name to its h-value
        self.heuristic: dict[str, float] = heuristic

    def h(self, node: str) -> float:
        """Return the heuristic value h(n) for a given node.
        Returns infinity if the node has no heuristic entry."""
        # Look up the heuristic value, default to infinity if not found
        return self.heuristic[node] if node in self.heuristic else float("inf")

    def g(self, path: tuple[str]) -> float:
        """Recursively compute the actual path cost g(n) for a given path.
        The path is a tuple of node names from start to current node."""
        # Base case: a single node has zero cost
        if len(path) < 2:
            return 0
        # Recursive case: cost = cost of remaining path + weight of first edge
        return (
            self.g(path[1:])  # Recursively compute cost of the rest of the path
            + next(
                edge  # Find the matching edge for the first two nodes in the path
                for edge in self.edges
                if edge[0] == path[0] and edge[1] == path[1]
            )[2]  # Extract the edge weight (third element of the tuple)
        )

    def expand(self, node: str) -> list[str]:
        """Return all neighbor nodes reachable from the given node."""
        # Collect target nodes from all edges originating at the given node
        return [edge[1] for edge in self.edges if edge[0] == node]

    def print_queue(self, queue: list[tuple], with_cost: bool = True, with_heuristic: bool = True) -> None:
        """Pretty-print the current state of the search queue.
        Shows each path with its g-cost and/or h-value."""
        print(
            "(",
            " ".join(
                # Format each path: concatenate node names, then append cost info
                f"{''.join(path)}"  # Join node names into a string (e.g., 'ACB')
                f"{'.' if with_cost or with_heuristic else ''}"  # Add separator dot if showing costs
                f"{str(self.g(path)) if with_cost else ''}"  # Append g-cost if requested
                f"{'+' if with_cost and with_heuristic else ''}"  # Add '+' between g and h
                f"{str(self.h(path[-1])) if with_heuristic else ''}"  # Append h-value if requested
                for path in queue  # Iterate over all paths in the queue
            ),
            ")",
        )

## 3. A* Search Function

In [11]:
def a_star(start: str, goal: str, graph: Graph) -> Optional[tuple]:
    """Perform A* search from start to goal on the given graph.
    Returns the optimal path as a tuple of node names, or None if no path exists."""

    # Initialize the queue with the start node as a single-element path
    queue: list[tuple] = [(start,)]
    # Track visited nodes to avoid revisiting them
    visited: set[str] = set()

    # Main loop: continue while there are paths to explore
    while queue:
        # Print the current state of the queue for visualization
        graph.print_queue(queue)

        # Pop the first (lowest f-cost) path from the queue
        current_path: tuple = queue.pop(0)
        # Get the last node in the current path
        current_node: str = current_path[-1]

        # Goal check: if we reached the goal, return the path
        if current_node == goal:
            return current_path

        # Mark the current node as visited
        visited.add(current_node)

        # Expand all neighbors of the current node
        for node in graph.expand(current_node):
            # Case 1: node is not visited and not already in the queue
            if node not in visited and node not in map(
                lambda x: x[-1], queue  # Extract the last node from each path in queue
            ):
                # Add the new path (current path + neighbor) to the queue
                queue.append(current_path + (node,))
                continue
            # Case 2: node is already in the queue — check if new path is cheaper
            elif node in map(lambda x: x[-1], queue):
                # Find the existing path in the queue that ends at this node
                existing_path = next(p for p in queue if p[-1] == node)
                # Compare g-costs: if the new path is cheaper, replace the old one
                if graph.g(current_path + (node,)) < graph.g(existing_path):
                    queue.remove(existing_path)  # Remove the old, more expensive path
                    queue.append(current_path + (node,))  # Add the new, cheaper path
                    continue

        # Sort the queue by f(n) = g(n) + h(n) so the best path is explored first
        queue.sort(
            key=lambda x: graph.g(x) + graph.h(x[-1])
        )

    # If the queue is exhausted without finding the goal, return None
    return None

## 4. Define the Graph

In [12]:
# Create a directed weighted graph with 5 nodes (A through E)
# Each edge tuple is (source_node, target_node, weight)
# Heuristic values estimate the remaining distance to the goal node E
graph: Graph = Graph(
    edges=[
        ("A", "B", 10),  # Edge from A to B with cost 10
        ("A", "C", 5),   # Edge from A to C with cost 5
        ("A", "D", 15),  # Edge from A to D with cost 15
        ("B", "D", 9),   # Edge from B to D with cost 9
        ("B", "E", 30),  # Edge from B to E with cost 30
        ("C", "B", 15),  # Edge from C to B with cost 15
        ("C", "D", 9),   # Edge from C to D with cost 9
        ("D", "B", 10),  # Edge from D to B with cost 10
        ("D", "E", 20),  # Edge from D to E with cost 20
    ],
    heuristic={
        "A": 30,  # Estimated distance from A to goal (E)
        "B": 25,  # Estimated distance from B to goal (E)
        "C": 20,  # Estimated distance from C to goal (E)
        "D": 15,  # Estimated distance from D to goal (E)
        "E": 0,   # Goal node — heuristic is 0
    },
)

## 5. Run A* Search

In [13]:
# Run A* search from node 'A' to goal node 'E'
result = a_star("A", "E", graph)

# Display the result
if result is not None:
    # Print the found path as a readable string (e.g., A -> C -> D -> E)
    print(f"\nOptimal path found: {' -> '.join(result)}")
    # Print the total cost of the optimal path
    print(f"Total path cost: {graph.g(result)}")
else:
    # No path was found between start and goal
    print("No path found from A to E.")

( A.0+30 )
( AC.5+20 AD.15+15 AB.10+25 )
( ACD.14+15 AB.10+25 )
( ACDE.34+0 AB.10+25 )

Optimal path found: A -> C -> D -> E
Total path cost: 34
